# ECC3479 Exploratory Data Analysis  
## AFL winning streaks and match attendance

This notebook is designed to sit inside the **existing ECC3479 project folder**.

It follows the unit's EDA logic:
- describe what the cleaned sample looks like,
- inspect variation within key variables,
- inspect covariation between the main outcome and main predictor,
- note anything that matters for later modelling.

**Expected data location:** `data/clean/afl_matches_with_streaks.csv`  
**Expected output location:** `output/eda/` (or `outputs/eda/` if that is already how your repo is organised)


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

sns.set_theme(style="whitegrid")

In [ ]:
def find_repo_root():
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve().parent.parent,
    ]
    for path in candidates:
        if (path / "data" / "clean" / "afl_matches_with_streaks.csv").exists():
            return path
    raise FileNotFoundError("Run this notebook from within your ECC3479 project folder.")

def choose_output_dir(repo_root):
    if (repo_root / "outputs").exists() and not (repo_root / "output").exists():
        out_dir = repo_root / "outputs" / "eda"
    else:
        out_dir = repo_root / "output" / "eda"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

repo_root = find_repo_root()
data_path = repo_root / "data" / "clean" / "afl_matches_with_streaks.csv"
out_dir = choose_output_dir(repo_root)

print("Repo root:", repo_root)
print("Data path:", data_path)
print("Output dir:", out_dir)

## 1. Load and inspect the cleaned data

Start by checking column names, types, missingness, and sample size.  
This is the first sanity check after cleaning and merging.


In [ ]:
df = pd.read_csv(data_path)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

if "match_date" in df.columns:
    df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")

for col in ["attendance", "max_pre_streak", "margin", "season", "is_finals"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "log_attendance" not in df.columns and "attendance" in df.columns:
    df["log_attendance"] = np.where(df["attendance"] > 0, np.log(df["attendance"]), np.nan)

df.head()

In [ ]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).values
}).sort_values(["missing_pct", "column"], ascending=[False, True])

overview

## 2. Build the EDA sample

The main descriptive relationship is between attendance and pre-match winning streak, so remove rows missing either of those values for the core analysis.


In [ ]:
analysis_df = df.dropna(subset=[c for c in ["attendance", "max_pre_streak"] if c in df.columns]).copy()

print(f"Original rows: {len(df):,}")
print(f"EDA rows: {len(analysis_df):,}")
print(f"Rows dropped for core EDA: {len(df) - len(analysis_df):,}")

## 3. Basic summary statistics

These summary statistics describe the sample's central tendency, spread, and range.


In [ ]:
summary_cols = [c for c in ["attendance", "log_attendance", "max_pre_streak", "margin", "season"] if c in analysis_df.columns]
summary_stats = analysis_df[summary_cols].describe().T
summary_stats

## 4. Univariate EDA

The first question is what the important variables look like on their own.


In [ ]:
if "attendance" in analysis_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(analysis_df["attendance"].dropna(), bins=30, ax=axes[0])
    axes[0].set_title("Distribution of attendance")
    axes[0].set_xlabel("Attendance")
    sns.boxplot(x=analysis_df["attendance"].dropna(), ax=axes[1])
    axes[1].set_title("Box plot of attendance")
    axes[1].set_xlabel("Attendance")
    plt.tight_layout()
    plt.show()

In [ ]:
if "log_attendance" in analysis_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(analysis_df["log_attendance"].dropna(), bins=30)
    plt.title("Distribution of log attendance")
    plt.xlabel("Log attendance")
    plt.tight_layout()
    plt.show()

if "max_pre_streak" in analysis_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(analysis_df["max_pre_streak"].dropna(), discrete=True)
    plt.title("Distribution of pre-match winning streaks")
    plt.xlabel("Max pre-match winning streak")
    plt.tight_layout()
    plt.show()

## 5. Time, finals, and venue patterns

These are important because they may confound the raw streak-attendance relationship.


In [ ]:
if {"season", "attendance"}.issubset(analysis_df.columns):
    season_att = (
        analysis_df.groupby("season", as_index=False)["attendance"]
        .mean()
        .rename(columns={"attendance": "mean_attendance"})
    )
    display(season_att.head())

    plt.figure(figsize=(9, 5))
    sns.lineplot(data=season_att, x="season", y="mean_attendance", marker="o")
    plt.title("Mean attendance by season")
    plt.xlabel("Season")
    plt.ylabel("Mean attendance")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"is_finals", "attendance"}.issubset(analysis_df.columns):
    finals_summary = (
        analysis_df.groupby("is_finals")["attendance"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )
    display(finals_summary)

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=analysis_df, x="is_finals", y="attendance")
    plt.title("Attendance by finals status")
    plt.xlabel("Is finals")
    plt.ylabel("Attendance")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"venue", "attendance"}.issubset(analysis_df.columns):
    top_venues = analysis_df["venue"].value_counts().head(10).index
    venue_df = analysis_df[analysis_df["venue"].isin(top_venues)].copy()

    venue_summary = (
        venue_df.groupby("venue")["attendance"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )
    display(venue_summary)

    plt.figure(figsize=(11, 6))
    sns.boxplot(data=venue_df, x="venue", y="attendance")
    plt.title("Attendance by major venue")
    plt.xlabel("Venue")
    plt.ylabel("Attendance")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Multivariate EDA: streak and attendance

This is the core exploratory relationship for the project.


In [ ]:
corr_rows = []
for y in [c for c in ["attendance", "log_attendance"] if c in analysis_df.columns]:
    subset = analysis_df[["max_pre_streak", y]].dropna()
    corr_rows.append({
        "outcome": y,
        "pearson_corr": subset["max_pre_streak"].corr(subset[y], method="pearson"),
        "spearman_corr": subset["max_pre_streak"].corr(subset[y], method="spearman"),
        "n": len(subset),
    })

corr_table = pd.DataFrame(corr_rows)
corr_table

In [ ]:
if {"max_pre_streak", "attendance"}.issubset(analysis_df.columns):
    plt.figure(figsize=(8, 5))
    sns.regplot(
        data=analysis_df,
        x="max_pre_streak",
        y="attendance",
        scatter_kws={"alpha": 0.4},
        line_kws={"linewidth": 2},
    )
    plt.title("Attendance and winning streak")
    plt.xlabel("Max pre-match winning streak")
    plt.ylabel("Attendance")
    plt.tight_layout()
    plt.show()

if {"max_pre_streak", "log_attendance"}.issubset(analysis_df.columns):
    plt.figure(figsize=(8, 5))
    sns.regplot(
        data=analysis_df,
        x="max_pre_streak",
        y="log_attendance",
        scatter_kws={"alpha": 0.4},
        line_kws={"linewidth": 2},
    )
    plt.title("Log attendance and winning streak")
    plt.xlabel("Max pre-match winning streak")
    plt.ylabel("Log attendance")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"max_pre_streak", "attendance"}.issubset(analysis_df.columns):
    streak_summary = (
        analysis_df.groupby("max_pre_streak")["attendance"]
        .agg(["count", "mean", "median"])
        .reset_index()
        .sort_values("max_pre_streak")
    )
    display(streak_summary)

    plt.figure(figsize=(9, 5))
    sns.lineplot(data=streak_summary, x="max_pre_streak", y="mean", marker="o", label="Mean")
    sns.lineplot(data=streak_summary, x="max_pre_streak", y="median", marker="o", label="Median")
    plt.title("Mean and median attendance by winning streak")
    plt.xlabel("Max pre-match winning streak")
    plt.ylabel("Attendance")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"max_pre_streak", "attendance", "is_finals"}.issubset(analysis_df.columns):
    plt.figure(figsize=(9, 5))
    sns.scatterplot(
        data=analysis_df,
        x="max_pre_streak",
        y="attendance",
        hue="is_finals",
        alpha=0.4,
    )
    plt.title("Attendance and streak by finals status")
    plt.xlabel("Max pre-match winning streak")
    plt.ylabel("Attendance")
    plt.tight_layout()
    plt.show()

## 7. A simple partial relationship check

This is **not** the final econometric analysis.  
It is just a useful EDA check of whether the raw streak pattern remains once season effects are absorbed.


In [ ]:
if {"season", "attendance", "max_pre_streak"}.issubset(analysis_df.columns):
    reg_df = analysis_df[analysis_df["attendance"] > 0].copy()
    m = smf.ols("np.log(attendance) ~ max_pre_streak + C(season)", data=reg_df).fit()
    display(pd.DataFrame({
        "term": m.params.index,
        "coef": m.params.values,
        "std_err": m.bse.values,
        "p_value": m.pvalues.values
    }))

## 8. What to write up from this notebook

Your written EDA should answer:

1. What does the sample look like?
2. Is attendance skewed, and does logging help?
3. Are winning streaks mostly short?
4. Does a first-order relationship exist between streaks and attendance?
5. Does the pattern differ by finals status, season, or venue?
6. What does this imply for the Week 9 regression?

A strong concluding sentence is:

> The EDA suggests that winning streaks are associated with attendance in the raw data, but the pattern is shaped by broader match characteristics, so the next stage should use a logged outcome and include sensible controls.
